# TSE 2026 — Fase 5T / opção Y

## Recebimento BWEB versus publicação EA20: auditoria cronológica condicionada

Investigamos **uma restrição necessária** do mecanismo operacional: se cada BU foi recebido antes de seus votos serem publicados e se os relógios usam a mesma referência BRT, então os totais nacionais publicados em cada instante não podem exceder os totais já recebidos naquele instante — considerando votos válidos, Lula e Flávio separadamente.

**NÃO** equiparamos um BU a um lote EA20; com os dados disponíveis isso não é possível. A desigualdade é somente **necessária**, não suficiente. Uma violação pode denunciar diferença de semântica dos timestamps, referência de relógios, domínio da coleta ou outro mecanismo; **não demonstra alteração de votos**.

A comparação BWEB×EA20 é condicional porque ambos são fontes oficiais, não relógios externos independentemente sincronizados. Serão apresentados controles de deslocamentos **predefinidos** de tempo para diagnosticar sensibilidade, não para escolher arbitrariamente uma correção.

Só primeiro turno de 2026. Todos os CSV de BWEB H e correções NR28 M tiveram seus SHA-256 conferidos na opção Y. Não usamos ZIP, rede, SQLite ou dados de 2022/2T.

## 1 — Abrir resultados da opção Y com verificação SHA e reconciliação

A opção Y gera a linha do tempo EA20 com contagens BWEB já recebidas em cada timestamp e controles de diferença fixa de relógios.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo():
    for p in (Path.cwd(),*Path.cwd().parents):
        if (p/"start_windows.bat").is_file():
            return p
    raise FileNotFoundError("Notebook fora da pasta do repositorio TSE")
REPO=find_repo()
sys.path.insert(0,str(REPO))
from urnas import root_path
BASE=root_path(None)/"analysis"/"periodicidade_bweb_2026"

def sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

summary_file=BASE/"phase5t_receipt_vs_publication_2026_summary.json"
q=json.loads(summary_file.read_text(encoding="utf-8"))
assert q["status"]=="BWEB_RECEBIMENTO_VS_PUBLICACAO_EA20_2026"
assert q["year"]==2026 and q["round"]==1
assert sha256(REPO/"data/forensics/analysis/phase5a_presidential_national_2026.csv")==q["source_sha256"]["EA20_national"]
def read_verified(meta):
    p=Path(meta["path"])
    if not p.is_file(): p=BASE/p.name
    assert sha256(p)==meta["sha256"], f"SHA-256 invalido: {p}"
    return pd.read_csv(p,sep=";",encoding="utf-8-sig")
timeline=read_verified(q["output_files"]["timeline"])
offsets=read_verified(q["output_files"]["offsets"])
timeline["date"]=pd.to_datetime(timeline["ea20_brt"])
assert len(timeline)==193 and q["source_events"]["sections"]==499206
assert q["source_events"]["valid"]==119300788
assert q["source_events"]["lula"]==53879538
assert q["source_events"]["flavio"]==56104503
print("OK: 499.206 secoes BWEB e 193 snapshots EA20, contagens finais reconciliadas.")
print("Eventos BWEB por segundo:",q["source_events"]["unique_bweb_seconds"])
print("Janela de recebimento:",q["source_events"]["first_bweb_second"],"a",q["source_events"]["last_bweb_second"])

## 2 — Condição necessária no relógio declarado, sem ajustar offsets

A diferença é **recebido cumulativo até t menos publicado EA20 em t** para cada contagem. Negativo viola a hipótese conjunta de relógio compatível, mesma população de votos e recebimento anterior à publicação. Positivo não prova que esses votos específicos estejam em uma fila.

In [ ]:
fig,ax=plt.subplots(figsize=(12,5))
for col,label in [
    ("valid_received_minus_published","Válidos"),
    ("lula_received_minus_published","Lula"),
    ("flavio_received_minus_published","Flávio"),
]:
    ax.plot(timeline["date"],timeline[col],label=label)
ax.axhline(0,linestyle="--")
ax.axvline(pd.Timestamp("2026-10-04 20:04:39"),linestyle=":",label="Corte EA20")
ax.set(title="Recebidos no BWEB até t − Publicados pelo EA20 em t",
       xlabel="Horário declarado BRT (comparação condicional)",
       ylabel="Diferença acumulada de votos")
ax.legend()
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
display(pd.DataFrame([q["baseline_offsets"]]).T)

## 3 — Onde a condição de causalidade agregada seria violada?

Contamos separadamente as violações para votos válidos, Lula e Flávio. Se a publicação exceder o recebido em alguma categoria, o modelo simples é incompatível com os timestamps **tal como declarados**. Isso, sozinho, não indica votos modificados.

In [ ]:
violations=timeline.loc[
    (timeline["valid_received_minus_published"]<0) |
    (timeline["lula_received_minus_published"]<0) |
    (timeline["flavio_received_minus_published"]<0)
].copy()
print("Snapshots com alguma violação no horário declarado:",len(violations),"/",len(timeline))
display(violations[[
    "ea20_brt","valid_received_minus_published",
    "lula_received_minus_published","flavio_received_minus_published"
]].head(30))
display(timeline.loc[timeline["ea20_brt"]=="2026-10-04 20:04:39",[
    "ea20_brt","valid_published","valid_received_by_then",
    "lula_published","lula_received_by_then",
    "flavio_published","flavio_received_by_then"
]])

## 4 — Controle de possíveis diferenças de referência de relógio

Os deslocamentos de BWEB de **0, ±5 minutos e ±3 horas** foram especificados no código antes de ler os resultados. Não são uma licença para escolher o offset mais conveniente. Valores negativos e positivos podem indicar, entre outras coisas, discrepância de timezone ou de semântica, exigindo confirmação documental independente.

In [ ]:
display(offsets[[
    "offset_seconds_applied_to_bweb",
    "snapshots_any_excess",
    "snapshots_published_exceeds_received_valid",
    "snapshots_published_exceeds_received_lula",
    "snapshots_published_exceeds_received_flavio",
    "max_valid_excess_votes",
    "max_lula_excess_votes",
    "max_flavio_excess_votes",
    "final_valid_received"
]])
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(offsets["offset_seconds_applied_to_bweb"].astype(str),offsets["snapshots_any_excess"])
ax.set(title="Sensibilidade das violações ao deslocamento hipotético do relógio BWEB",
       xlabel="Offset de segundos aplicado aos timestamps BWEB",
       ylabel="Capturas EA20 com pelo menos uma violação")
ax.grid(axis="y",alpha=.2)
plt.show()

## 5 — Limite por posição cumulativa: não é atraso por BU

Para cada snapshot EA20, perguntamos **qual a primeira data em que havia sido recebido um total suficiente** de válidos, Lula e Flávio, simultaneamente. Usamos a mais tardia dessas três datas como limite inferior de disponibilidade agregada. O campo `aggregate_rank_slack_seconds` compara o instante de publicação com esse limite.

Isso **não** mede a espera de cada BU individual. Correspondência por posição acumulada não equivale a identidade de votos nem reconstitui fila interna.

In [ ]:
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(timeline["date"],timeline["aggregate_rank_slack_seconds"]/60)
ax.axhline(0,linestyle="--")
ax.axvline(pd.Timestamp("2026-10-04 20:04:39"),linestyle=":")
ax.set(title="Folga temporal entre publicar e disponibilidade acumulada mínima",
       xlabel="Horário da publicação EA20",
       ylabel="Segundos convertidos em minutos (não é espera individual)")
ax.grid(alpha=.2)
fig.autofmt_xdate()
plt.show()
display(timeline.nsmallest(15,"aggregate_rank_slack_seconds")[[
    "ea20_brt","latest_necessary_receipt_time","aggregate_rank_slack_seconds",
    "valid_received_minus_published","lula_received_minus_published",
    "flavio_received_minus_published"
]])

## 6 — Antes e depois do corte: há alteração operacional mensurável?

Resumimos os resultados em dois períodos sem reinterpretar a desigualdade como causa. A fase X constatou que o volume publicado em cada atualização é irregular; o gráfico abaixo examina se a diferença entre etapas de processamento muda no mesmo período.

In [ ]:
timeline["periodo"]=timeline["date"].map(
    lambda t:"Antes do corte" if t<pd.Timestamp("2026-10-04 20:04:39") else "A partir do corte"
)
table=timeline.groupby("periodo",as_index=False).agg(
    snapshots=("ea20_brt","count"),
    violations=("all_three_constraints_ok",lambda x:int((~x.astype(bool)).sum())),
    min_valid_gap=("valid_received_minus_published","min"),
    max_valid_gap=("valid_received_minus_published","max"),
    min_lula_gap=("lula_received_minus_published","min"),
    min_flavio_gap=("flavio_received_minus_published","min"),
)
display(table)

## 7 — Critérios de interpretação e próximos passos

**Se houver déficits de votos recebidos:** verificar primeiro referências de relógio, significado exato do campo `DT_BU_RECEBIDO`, eventuais eventos de transmissão/publicação e a qualidade das capturas antes de atribuir uma anomalia ao resultado eleitoral.

**Se a desigualdade for satisfeita:** a etapa simples de recepção antes da publicação é viável em termos agregados; isso não estabelece quais seções foram publicadas em cada incremento nem prova a existência de buffering.

**Qualquer resultado:** a próxima evidência decisiva requer snapshots EA20 por UF ou município **sincronizados e preservados**, ou registros internos de publicação/transmissão com identificadores de seção e timestamps auditáveis. Só então será possível testar a composição e a lisura dos lotes nacionais sem imputar eventos que não observamos.

Não concluímos fraude nem sua impossibilidade com uma curva acumulada.

In [ ]:
print("Versão:",q["version"])
print("Status:",q["status"])
print("Timestamps por segundo preservados:",q["source_events"]["unique_bweb_seconds"])
print("Condição de relógio declarado satisfeita em todos snapshots:",
      q["baseline_offsets"]["all_constraints_held"])
print("Violações da hipótese conjunta:",q["baseline_offsets"]["snapshot_violations"])
print("CSV SHA:",q["output_files"]["timeline"]["sha256"])
print("Dados originais intactos.")